In [1]:
import os
os.chdir('../')

In [6]:
import torch
import torch.nn as nn
from config import GPTConfig
from tokenizer import BPETokenizer
from model.attention import MultiHeadAttention

In [7]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, epsilon=1e-5):
        super().__init__()
        
        self.epsilon = epsilon
        self.emb_dim = emb_dim
        self.scale = nn.Parameter(torch.ones(self.emb_dim))
        self.shift = nn.Parameter(torch.zeros(self.emb_dim))
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean)/torch.sqrt(variance + self.epsilon)
        return self.scale * x_norm + self.shift

In [8]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
                            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
                            (x + 0.044715 * torch.pow(x, 3))
                            ))

In [9]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            GELU(),
            nn.Linear(4 * config.d_model, config.d_model)
        )
        
    def forward(self, x):
        return self.layers(x)

In [10]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        
        self.attn = MultiHeadAttention(d_in = config.d_model,
                                       d_out = config.d_model,
                                       context_length = config.context_length,
                                       dropout = config.dropout,
                                       num_heads = config.n_heads,
                                       qkv_bias = config.qkv_bias)
        
        self.ff = FeedForward(config)
        self.norm1 = LayerNorm(config.d_model)
        self.norm2 = LayerNorm(config.d_model)
        self.drop_shortcut = nn.Dropout(config.dropout)
    
    def forward(self, x):
        
        shortcut = x
        x = self.norm1(x)
        x = self.attn(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        
        return x

In [11]:
class LLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.context_length, config.d_model)
        self.drop_emb = nn.Dropout(config.dropout)
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(config) for _ in range(config.n_layers)]
            )
        self.final_norm = LayerNorm(config.d_model)
        self.output_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        
    
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.output_head(x)
        return logits

In [12]:
tokenizer = BPETokenizer()
tokenizer.load("data/vocab.json", "data/merges.json")

In [13]:
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[2505, 6708, 6782,  367],
        [2505,  369, 4294,  270]])


In [14]:
torch.manual_seed(123)

model = LLM(GPTConfig())
logits = model(batch)

print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 8192])
tensor([[[-0.4196,  0.9413, -0.0932,  ...,  0.4545,  0.5521, -0.1376],
         [ 0.4897,  0.4568,  0.8544,  ..., -0.5710, -0.1774,  0.1102],
         [ 0.7885, -0.1467, -0.7006,  ..., -0.1808,  0.0200,  0.8759],
         [ 0.0441,  0.0218,  0.7052,  ..., -0.4420,  0.3201,  0.0647]],

        [[-0.3327,  1.0971, -0.4249,  ...,  0.6135,  0.2243, -0.0399],
         [-1.0537, -0.4329, -0.2674,  ...,  0.5499,  0.6433,  0.1668],
         [-1.1466, -0.1863, -0.3368,  ..., -0.1018,  0.2448, -0.1543],
         [ 0.4018,  0.5817,  0.9332,  ...,  0.2908, -0.6287,  0.6539]]],
       grad_fn=<UnsafeViewBackward0>)


In [15]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 33,727,488


In [17]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.output_head.weight.shape)

Token embedding layer shape: torch.Size([8192, 512])
Output layer shape: torch.Size([8192, 512])


In [18]:
total_size_bytes = total_params * 4
total_size_mb = total_size_bytes / (1024 * 1024)
print(f"Total size of the model: {total_size_mb:.2f} MB")


Total size of the model: 128.66 MB


In [19]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [20]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [72, 1835, 350, 1258]
encoded_tensor.shape: torch.Size([1, 4])


In [23]:
model.eval()
out = generate_text_simple(
 model=model,
 idx=encoded_tensor,
 max_new_tokens=6,
 context_size=GPTConfig().context_length
)
print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[  72, 1835,  350, 1258, 7592, 5818, 2538, 2503, 5597, 1205]])
Output length: 10


In [24]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am Mom's ears front mach rescue ask
